# Semana 12 · Sistemas multi-agente (experimento del estudiante)

Mismas secciones que la demo, con predicciones antes de ejecutar y TODOs. El notebook corre de principio a fin aunque los TODO queden sin llenar; llenarlos es la actividad.

## 0. Preparación

In [ ]:
# En Colab: instala el SDK de agentes y lo que usan las tools. En local (.venv) ya está todo; la salida se silencia.
!pip install -q openai openai-agents pydantic transformers torch sentence-transformers > /dev/null 2>&1; echo "dependencias listas"

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
# Backend: llave de OpenAI si existe; si no, servidor local compatible (transformers serve).
# Copia reducida de course/shared/local_server.py para que el notebook sea autocontenido.
import os, shutil, subprocess, sys, time, urllib.request

MODELO_LOCAL, PUERTO = "Qwen/Qwen2.5-1.5B-Instruct", 8765

def _vivo(raiz):
    try:
        with urllib.request.urlopen(f"{raiz}/health", timeout=2) as r: return r.status == 200
    except Exception: return False

def ensure_local_server(modelo=MODELO_LOCAL, puerto=PUERTO, espera_s=900):
    raiz = f"http://localhost:{puerto}"
    if _vivo(raiz): return raiz + "/v1"
    exe = shutil.which("transformers") or os.path.join(os.path.dirname(sys.executable), "transformers")
    subprocess.Popen([exe, "serve", "--port", str(puerto), modelo], stdout=open("/tmp/serve.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < espera_s:
        if _vivo(raiz): return raiz + "/v1"
        time.sleep(3)
    raise RuntimeError("el servidor local no arrancó; revisa /tmp/serve.log")

def backend_config():
    if os.environ.get("OPENAI_API_KEY") and not os.environ.get("OPENAI_BASE_URL"):
        return {"base_url": None, "api_key": os.environ["OPENAI_API_KEY"], "model": os.environ.get("OPENAI_MODEL", "gpt-5-mini"), "local": False}
    base = os.environ.get("OPENAI_BASE_URL") or ensure_local_server()
    return {"base_url": base, "api_key": "local", "model": MODELO_LOCAL, "local": True}

CFG = backend_config()
print("backend:", "servidor local (Chat Completions)" if CFG["local"] else "OpenAI (Responses API)", "| modelo:", CFG["model"])

In [ ]:
# Tools de negocio de Facturio: shared/facturio_tools.py. En local se importa; en Colab se descarga.
import sys, urllib.request
from pathlib import Path
CANDIDATOS = [Path("../../shared"), Path("../shared"), Path("shared")]
ruta = next((p for p in CANDIDATOS if (p / "facturio_tools.py").exists()), None)
if ruta is None:
    url = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/facturio_tools.py"
    Path("shared").mkdir(exist_ok=True)
    urllib.request.urlretrieve(url, "shared/facturio_tools.py")
    ruta = Path("shared")
sys.path.insert(0, str(ruta.resolve()))
import facturio_tools as ft
FOLIO, DUP = list(ft.CARGOS)[0], "PAY-00012348"
print("tools de negocio:", list(ft.TOOLS), "| folio de ejemplo:", FOLIO)

In [ ]:
# Tracing de consola (semana 11) más la ENTRADA de cada llamada al modelo: sirve para medir qué recibió cada agente.
# Con la llave, el SDK envía los spans al dashboard de OpenAI; aquí se sustituye el exportador.
import json
from datetime import datetime
from agents.tracing import TracingProcessor, set_trace_processors, set_tracing_disabled

def _seg(span):
    try: return (datetime.fromisoformat(span.ended_at) - datetime.fromisoformat(span.started_at)).total_seconds()
    except Exception: return 0.0

class TracingConsola(TracingProcessor):
    def __init__(self): self.traces = []; self._actual = None
    def on_trace_start(self, trace): self._actual = {"nombre": trace.name, "spans": []}
    def on_trace_end(self, trace): self.traces.append(self._actual); self._actual = None
    def on_span_start(self, span): pass
    def on_span_end(self, span):
        d, tipo, usage, entrada = span.span_data, span.span_data.type, {}, []
        if tipo == "generation":
            usage, entrada = d.usage or {}, list(d.input or [])
            texto = f"generation  {_seg(span):5.2f} s  in={usage.get('input_tokens',0)} out={usage.get('output_tokens',0)}"
        elif tipo == "function": texto = f"function {d.name}({str(d.input)[:40]}) -> {str(d.output)[:40]}"
        elif tipo == "handoff": texto = f"handoff {d.from_agent} -> {d.to_agent}"
        elif tipo == "agent": texto = f"agent {d.name}  {_seg(span):5.2f} s"
        else: texto = f"{tipo}  {_seg(span):5.2f} s"
        if self._actual is not None:
            self._actual["spans"].append({"tipo": tipo, "id": span.span_id, "padre": span.parent_id, "usage": usage, "entrada": entrada,
                                          "texto": texto + ("  [ERROR]" if span.error else ""), "nombre": getattr(d, "name", None), "seg": _seg(span)})
    def force_flush(self): pass
    def shutdown(self): pass
    def ultimo(self): return self.traces[-1] if self.traces else None
    def imprimir(self):
        t = self.ultimo(); print("RUN", t["nombre"])
        hijos = {}
        for s in t["spans"]: hijos.setdefault(s["padre"], []).append(s)
        def rec(p, n):
            for i, s in enumerate(hijos.get(p, [])):
                print("    " * n + ("└── " if i == len(hijos[p]) - 1 else "├── ") + s["texto"]); rec(s["id"], n + 1)
        rec(None, 0)
    def resumen(self):
        t = self.ultimo(); g = [s for s in t["spans"] if s["tipo"] == "generation"]
        return {"llamadas_modelo": len(g), "input_tokens": sum(s["usage"].get("input_tokens", 0) for s in g),
                "output_tokens": sum(s["usage"].get("output_tokens", 0) for s in g),
                "tool_calls": [s["nombre"] for s in t["spans"] if s["tipo"] == "function"],
                "latencia_s": round(max((s["seg"] for s in t["spans"] if s["tipo"] == "agent"), default=0), 2)}

def contexto_que_viajo(trace):
    """Por agente: mensajes, caracteres, tokens de entrada y roles de su PRIMERA llamada al modelo."""
    por_id = {s["id"]: s for s in trace["spans"]}; salida = {}
    for s in trace["spans"]:
        if s["tipo"] != "generation": continue
        p, agente = s["padre"], None
        while p in por_id:
            if por_id[p]["tipo"] == "agent": agente = por_id[p]["nombre"]; break
            p = por_id[p]["padre"]
        if agente in salida: continue
        salida[agente] = {"mensajes": len(s["entrada"]), "caracteres": sum(len(json.dumps(m, ensure_ascii=False)) for m in s["entrada"]),
                          "input_tokens": s["usage"].get("input_tokens", 0), "roles": [m.get("role", m.get("type", "?")) for m in s["entrada"]]}
    return salida

set_tracing_disabled(False)
consola = TracingConsola()
set_trace_processors([consola])      # nada viaja a OpenAI; con la llave, quitar esta línea para usar el dashboard
print("tracing de consola activo")

In [ ]:
# Los especialistas y las cinco topologías (la versión completa está en examples/topologias.py).
import asyncio, dataclasses, re, time
from pydantic import BaseModel, Field
from agents import Agent, Runner, function_tool, handoff, HandoffInputData, ModelSettings, OpenAIChatCompletionsModel
from agents.extensions import handoff_filters

def modelo():
    if CFG["local"]:                                     # el servidor local solo habla Chat Completions
        from openai import AsyncOpenAI
        return OpenAIChatCompletionsModel(model=CFG["model"], openai_client=AsyncOpenAI(base_url=CFG["base_url"], api_key="local", timeout=300))
    return CFG["model"]                                  # con llave: Responses API por defecto

AJUSTES = ModelSettings(max_tokens=160, temperature=0)
FALLAS = {"consultar_factura": False}                    # para provocar una caída controlada

@function_tool(failure_error_function=None)              # None: la excepción sube en lugar de volverse texto para el modelo
def consultar_factura(folio: str) -> str:
    """Consulta una factura por folio (formato PAY- seguido de 8 dígitos). Devuelve estado, monto y fecha."""
    if FALLAS["consultar_factura"]: raise RuntimeError("servicio de cobros no disponible (timeout)")
    return json.dumps(ft.consultar_factura(folio), ensure_ascii=False)

@function_tool
def consultar_plan(usuario_id: str) -> str:
    """Consulta el plan de un usuario (u001 a u005): límites, uso actual y precio."""
    return json.dumps(ft.consultar_plan(usuario_id), ensure_ascii=False)

@function_tool
def buscar_documentacion(pregunta: str) -> str:
    """Busca en la documentación y políticas de Facturio. Úsala para dudas de políticas, límites y errores."""
    return json.dumps(ft.buscar_documentacion(pregunta, k=2), ensure_ascii=False)

@function_tool(needs_approval=True)
def reembolsar(folio: str, monto: float, motivo: str) -> str:
    """Solicita un reembolso sobre un cargo. Tiene side effect: mueve dinero. Requiere aprobación humana."""
    ok, razon = ft.autorizar_reembolso(folio, monto)
    return json.dumps(ft.reembolsar(folio, monto, motivo) if ok else {"ejecutado": False, "razon": razon}, ensure_ascii=False)

@function_tool
def escalar_a_humano(motivo: str) -> str:
    """Pasa el caso a un agente humano cuando la política lo exige o falta información."""
    return json.dumps(ft.escalar_a_humano(motivo), ensure_ascii=False)

TODAS = [consultar_factura, consultar_plan, buscar_documentacion, reembolsar, escalar_a_humano]
PERMISOS = {"Facturacion": [consultar_factura, consultar_plan, reembolsar],      # least privilege: la lista es el permiso
            "Acceso": [buscar_documentacion],
            "Tecnico": [buscar_documentacion, escalar_a_humano]}
DESCRIPCION = {"Facturacion": "Cobros, reembolsos, cargos duplicados, planes y precios.",
               "Acceso": "Contraseñas, inicio de sesión, 2FA, bloqueos, invitaciones.",
               "Tecnico": "Errores E-xxxx, exportación, API, integraciones, fallas del producto."}
INSTRUCCIONES = {
    "Soporte": "Eres el asistente de la mesa de soporte de Facturio. Usa las tools cuando haga falta: consulta facturas y planes antes de afirmar algo, busca la documentación para políticas y errores, y escala a humano cuando falte información. Responde en español, breve.",
    "Facturacion": "Eres el especialista de facturación de Facturio. Usa consultar_factura o consultar_plan antes de afirmar algo sobre un cargo o un plan. Responde en español, breve.",
    "Acceso": "Eres el especialista de acceso y seguridad de Facturio. Usa buscar_documentacion antes de responder. Responde en español, breve.",
    "Tecnico": "Eres el especialista técnico de Facturio. Usa buscar_documentacion con el código de error antes de responder; si no hay solución documentada, escala a humano. Responde en español, breve.",
    "Triage": "Eres el triage de la mesa de soporte de Facturio. No respondas tú: transfiere cada ticket al especialista adecuado con la tool transfer_to_*.",
    "Orquestador": "Eres la mesa de soporte de Facturio. Para resolver el ticket llama a la tool del especialista adecuado y devuelve su respuesta al cliente. Responde en español, breve.",
    "Manager": "Eres el coordinador de la mesa de soporte de Facturio. El ticket puede tener varios temas: llama a la tool del especialista de cada tema (una llamada por tema) y al final integra las respuestas en un solo mensaje breve en español.",
}

def especialista(nombre):
    return Agent(name=nombre, handoff_description=DESCRIPCION[nombre], instructions=INSTRUCCIONES[nombre], tools=list(PERMISOS[nombre]), model=modelo(), model_settings=AJUSTES)
def especialistas(): return {n: especialista(n) for n in PERMISOS}

# ---- handoff package: lo que recibe el agente B cuando no se le manda el historial
class PaqueteHandoff(BaseModel):
    objective: str = Field(description="Qué necesita el cliente, en una oración")
    findings: list[str] = Field(default_factory=list, description="Datos ya conocidos: folios, usuario, códigos, fechas")
    sources: list[str] = Field(default_factory=list, description="De dónde salió cada dato (tool, documento, cliente)")
    constraints: list[str] = Field(default_factory=list, description="Límites que el especialista debe respetar")
    open_questions: list[str] = Field(default_factory=list, description="Lo que falta por averiguar")

PAQUETES = []
async def registrar_paquete(ctx, paquete: PaqueteHandoff):        # on_handoff: el código ve el paquete antes que el especialista
    p = paquete.model_dump()
    for f in re.findall(r"PAY-\d{8}", p["objective"] + " ".join(p["findings"])):
        if f not in p["findings"]: p["findings"].append(f)         # el código completa lo que el modelo omitió
    PAQUETES.append(p)

def filtro_paquete(data: HandoffInputData) -> HandoffInputData:    # input_filter: el especialista recibe solo el paquete
    args = next((getattr(i.raw_item, "arguments", "{}") for i in data.new_items if type(i).__name__ == "HandoffCallItem"), "{}") or "{}"
    try: paquete = PaqueteHandoff.model_validate_json(args).model_dump()
    except Exception: paquete = {"objective": args}
    for f in re.findall(r"PAY-\d{8}", args):
        if f not in paquete.get("findings", []): paquete.setdefault("findings", []).append(f)
    mensaje = {"role": "user", "content": "Handoff package:\n" + json.dumps(paquete, ensure_ascii=False, indent=1)}
    return dataclasses.replace(data, input_history=(mensaje,), pre_handoff_items=(), new_items=(), input_items=None)

FILTROS = {"completo": None, "sin_tools": handoff_filters.remove_all_tools, "paquete": filtro_paquete}

# ---- las cinco topologías
def single_agent():
    return Agent(name="Soporte", instructions=INSTRUCCIONES["Soporte"], tools=list(TODAS), model=modelo(), model_settings=AJUSTES)

def handoffs(contexto="sin_tools"):
    esp = especialistas()
    if contexto == "paquete":
        hs = [handoff(a, input_type=PaqueteHandoff, on_handoff=registrar_paquete, input_filter=filtro_paquete) for a in esp.values()]
    else:
        hs = [handoff(a, input_filter=FILTROS[contexto]) for a in esp.values()]
    return Agent(name="Triage", instructions=INSTRUCCIONES["Triage"], handoffs=hs, model=modelo(), model_settings=AJUSTES)

def _como_tools(esp):
    return [esp["Facturacion"].as_tool("preguntar_facturacion", "Resuelve dudas de cobros, reembolsos, planes y facturas. Pásale el folio o el usuario."),
            esp["Acceso"].as_tool("preguntar_acceso", "Resuelve dudas de contraseñas, sesión, 2FA, bloqueos e invitaciones."),
            esp["Tecnico"].as_tool("preguntar_tecnico", "Resuelve errores E-xxxx, exportación, API e integraciones.")]

def agents_as_tools():
    return Agent(name="Orquestador", instructions=INSTRUCCIONES["Orquestador"], tools=_como_tools(especialistas()), model=modelo(), model_settings=AJUSTES)

def manager():
    return Agent(name="Manager", instructions=INSTRUCCIONES["Manager"], tools=_como_tools(especialistas()), model=modelo(), model_settings=AJUSTES)

PALABRAS = {"Facturacion": ("factura", "cobr", "reembols", "plan ", "plan.", "precio", "pago", "cargo", "descuento"),
            "Acceso": ("contraseña", "sesión", "sesion", "2fa", "bloque", "invitaci", "acceso", "entrar"),
            "Tecnico": ("error", "e-", "export", "pdf", "api", "webhook", "integraci", "csv")}

def ruta_determinista(ticket):
    """El routing de la semana 8 sin modelo. None cuando el código no sabe; entonces atiende el agente general."""
    if re.search(r"PAY-\d{8}", ticket): return "Facturacion"
    if re.search(r"E-\d{4}", ticket): return "Tecnico"
    t = ticket.lower(); puntos = {n: sum(1 for k in ks if k in t) for n, ks in PALABRAS.items()}
    mejor = max(puntos, key=puntos.get)
    return mejor if puntos[mejor] > 0 else None

def routing_determinista(ticket):
    ruta = ruta_determinista(ticket)
    return (ruta, especialista(ruta)) if ruta else ("Soporte", single_agent())

TOPOLOGIAS = ["single_agent", "handoffs", "agents_as_tools", "manager", "routing_determinista"]
CONSTRUIR = {"single_agent": single_agent, "handoffs": handoffs, "agents_as_tools": agents_as_tools, "manager": manager}

async def correr(agente, entrada, aprobar=None, max_turns=6):
    """Runner.run más el ciclo de aprobación de la semana 11 (reembolsar tiene needs_approval)."""
    r = await Runner.run(agente, entrada, max_turns=max_turns)
    while getattr(r, "interruptions", None):
        estado = r.to_state()
        for item in r.interruptions:
            args = json.loads(item.raw_item.arguments or "{}")
            (estado.approve if (aprobar and aprobar(item.tool_name, args)) else estado.reject)(item)
        r = await Runner.run(agente, estado, max_turns=max_turns)
    return r

async def ejecutar(topologia, ticket, contexto="sin_tools", max_turns=6):
    """Corre un ticket con una topología y devuelve una fila medible."""
    PAQUETES.clear(); t0 = time.perf_counter(); ruta = None
    if topologia == "routing_determinista": ruta, agente = routing_determinista(ticket)
    else: agente = CONSTRUIR[topologia](contexto) if topologia == "handoffs" else CONSTRUIR[topologia]()
    try:
        r = await correr(agente, ticket, max_turns=max_turns); respuesta, ultimo, error = str(r.final_output or ""), r.last_agent.name, None
    except Exception as e:
        respuesta, ultimo, error = "", "?", f"{type(e).__name__}: {str(e)[:80]}"
    s = consola.resumen(); tools = s["tool_calls"]
    consultados = [t.replace("preguntar_", "").capitalize() for t in tools if t.startswith("preguntar_")]
    atendio = {"handoffs": ultimo, "agents_as_tools": "+".join(consultados) or ultimo, "manager": "+".join(consultados) or ultimo,
               "routing_determinista": ruta, "single_agent": "Soporte"}[topologia]
    return {"topologia": topologia, "atendio": atendio, "llamadas": s["llamadas_modelo"], "tools": [t for t in tools if not t.startswith("preguntar_")],
            "tokens": s["input_tokens"] + s["output_tokens"], "latencia_s": round(time.perf_counter() - t0, 2), "respuesta": respuesta, "error": error}

GOLD = [("g1", f"¿Cuál es el estado de la factura {FOLIO}?", "Facturacion", "consultar_factura"),
        ("g2", "Consulta el plan del usuario u002 y dime cuántas facturas al mes incluye.", "Facturacion", "consultar_plan"),
        ("g3", f"El cargo {DUP} me aparece duplicado, ¿me lo van a devolver?", "Facturacion", "consultar_factura"),
        ("g4", "No puedo iniciar sesión: el código 2FA nunca llega.", "Acceso", "buscar_documentacion"),
        ("g5", "Al exportar a PDF me sale el error E-4001.", "Tecnico", "buscar_documentacion"),
        ("g6", "Invité a un colega hace una semana y dice que el enlace de invitación ya no sirve.", "Acceso", "buscar_documentacion")]
TICKET_LARGO = ("Hola, buenas tardes. Les escribo porque llevo tres años con Facturio y en general estoy contento, aunque el mes pasado cambiamos de "
                f"contador y hubo confusión con los pagos. Mi contador dice que ya se pagó pero yo no veo nada en el portal. Mi usuario es u001 y el folio "
                f"del cargo es {FOLIO}. ¿Me pueden decir cuál es el estado de esa factura? Gracias de antemano, y disculpen la molestia.")
print("topologías listas:", TOPOLOGIAS)

## 1. Predicciones

Antes de correr nada. El ticket es `¿Cuál es el estado de la factura PAY-...?`. Con el agente único costó 2 llamadas al modelo en la semana 11.

In [ ]:
MI_PREDICCION_LLAMADAS = {"handoffs": None, "agents_as_tools": None, "routing_determinista": None}   # TODO: llamadas al modelo por ticket
MI_PREDICCION_CONTEXTO = None    # TODO: con la política "completo", ¿el especialista llamará a consultar_factura? True / False
MI_PREDICCION_FALLA = ""         # TODO: una línea: si consultar_factura se cae dentro de un agents-as-tools, ¿qué recibe el cliente?
print("predicciones:", MI_PREDICCION_LLAMADAS, MI_PREDICCION_CONTEXTO, MI_PREDICCION_FALLA)

## 2. Handoffs y el span de transferencia

In [ ]:
f = await ejecutar("handoffs", GOLD[0][1])
print(f); consola.imprimir()
DATOS_DEL_ARBOL = {"agentes": [], "llamadas_modelo": None, "tools": [], "quien_respondio": None}   # TODO: a partir del árbol

## 3. Qué contexto viajó

Compara con `MI_PREDICCION_CONTEXTO`. Después, modifica `registrar_paquete` para que el código agregue una `constraint` fija (el límite de aprobación de reembolsos, `ft.MONTO_MAXIMO_SIN_HUMANO`) y vuelve a correr la política `paquete`.

In [ ]:
# Handoff con tres políticas de contexto sobre el mismo ticket corto: qué recibió el especialista en su primera llamada.
for politica in FILTROS:
    f = await ejecutar("handoffs", GOLD[0][1], contexto=politica)
    viajo = contexto_que_viajo(consola.ultimo())
    esp = {k: v for k, v in viajo.items() if k != "Triage"}
    print(f"{politica:9s} atendió={f['atendio']:12s} llamadas={f['llamadas']} tools={f['tools']} tokens totales={f['tokens']}")
    print(f"          triage in={viajo.get('Triage', {}).get('input_tokens')} | especialista: {esp}")
    if PAQUETES: print("          paquete:", json.dumps(PAQUETES[0], ensure_ascii=False))

In [ ]:
# TODO: agrega la constraint del límite de aprobación al paquete desde el código (on_handoff) y vuelve a correr.
async def registrar_paquete(ctx, paquete: PaqueteHandoff):
    p = paquete.model_dump()
    for f in re.findall(r"PAY-\d{8}", p["objective"] + " ".join(p["findings"])):
        if f not in p["findings"]: p["findings"].append(f)
    # p["constraints"].append(...)        # TODO
    PAQUETES.append(p)

f = await ejecutar("handoffs", GOLD[0][1], contexto="paquete")
print("paquete que viajó:", json.dumps(PAQUETES[0], ensure_ascii=False) if PAQUETES else "(el triage no transfirió)")
OBSERVACION_CONTEXTO = ""   # TODO: qué cambió entre completo, sin_tools y paquete, con un dato del trace

## 4. Agents-as-tools y routing determinista

In [ ]:
for topo in ["agents_as_tools", "routing_determinista"]:
    f = await ejecutar(topo, GOLD[0][1]); print(f); consola.imprimir(); print()
print("predicción de llamadas:", MI_PREDICCION_LLAMADAS)

## 5. Actividad 1: topología y handoff package

Escenario en `exercises/01-topologia-y-handoff-package.md`. Escribe la topología con dos razones y el paquete para el especialista de facturación. Marca en `LLENA_EL_CODIGO` qué campos completa el código.

In [ ]:
TOPOLOGIA_ELEGIDA = ""            # TODO
RAZONES = ["", ""]                # TODO: dos razones ligadas a las restricciones del escenario
PAQUETE_ACTIVIDAD = {"objective": "", "findings": [], "sources": [], "constraints": [], "open_questions": []}   # TODO
LLENA_EL_CODIGO = []              # TODO: campos que completa el código antes de transferir
QUE_NO_VIAJA_A_ACCESO = ""        # TODO: qué información no viaja al especialista de acceso y qué mecanismo lo garantiza
print(json.dumps(PAQUETE_ACTIVIDAD, ensure_ascii=False, indent=1))

## 6. Tabla comparativa (una corrida por ticket; sube CORRIDAS si el backend lo permite)

In [ ]:
# Comparación medida: 1 corrida(s) por ticket y topología. Mismo modelo, mismos especialistas, mismas tools.
import statistics
CORRIDAS = 1
filas = []
for gid, ticket, esperado, tool in GOLD:
    for topo in TOPOLOGIAS:
        for k in range(CORRIDAS):
            ft.reiniciar_estado()
            f = await ejecutar(topo, ticket)
            f.update({"ticket": gid, "ruta_ok": None if topo == "single_agent" else esperado in str(f["atendio"]).split("+"), "tool_ok": tool in f["tools"]})
            filas.append(f)
            print(f"{gid} {topo:20s} #{k+1} atendió={str(f['atendio']):13s} ruta_ok={f['ruta_ok']} tool_ok={f['tool_ok']} llamadas={f['llamadas']} tok={f['tokens']} {f['latencia_s']}s err={f['error']}")

def resumen(topo):
    fs = [f for f in filas if f["topologia"] == topo]; con_ruta = [f for f in fs if f["ruta_ok"] is not None]
    return {"topología": topo, "ruta_ok": round(sum(f["ruta_ok"] for f in con_ruta) / len(con_ruta), 2) if con_ruta else "n/a",
            "tool_ok": round(sum(f["tool_ok"] for f in fs) / len(fs), 2), "llamadas": round(statistics.mean(f["llamadas"] for f in fs), 2),
            "tokens": round(statistics.mean(f["tokens"] for f in fs)), "latencia_s": round(statistics.mean(f["latencia_s"] for f in fs), 2),
            "errores": sum(1 for f in fs if f["error"])}
print()
for topo in TOPOLOGIAS: print(resumen(topo))

## 7. Falla provocada

Compara con `MI_PREDICCION_FALLA`.

In [ ]:
# La tool consultar_factura se cae. El mismo ticket, cuatro topologías: ¿dónde aparece la falla y quién decide qué pasa?
FALLAS["consultar_factura"] = True
for topo in ["single_agent", "handoffs", "agents_as_tools"]:
    f = await ejecutar(topo, GOLD[0][1])
    print(f"\n[{topo}] error={f['error']} | llamadas={f['llamadas']} | respuesta={f['respuesta'][:90]!r}")
    consola.imprimir()

# Routing determinista con fallback en código: la decisión no pasa por el modelo.
ruta, agente = routing_determinista(GOLD[0][1])
try:
    r = await correr(agente, GOLD[0][1]); print("\n[routing] respuesta:", r.final_output)
except Exception as e:
    t = ft.escalar_a_humano(f"falla en {ruta}: {type(e).__name__}", folio=FOLIO)
    print(f"\n[routing + fallback] la app atrapó {type(e).__name__} y abrió {t['ticket']}; el cliente recibe una explicación y un folio de seguimiento.")
FALLAS["consultar_factura"] = False

In [ ]:
OBSERVACION_FALLA = ""   # TODO: en qué topología decidió el modelo, en cuál la aplicación y en cuál el código

## 8. Actividad 2: ¿la mesa de soporte necesita más de un agente?

Máximo 120 palabras, con una topología, un dato de la tabla de la sección 6 y la condición que cambiaría la recomendación.

In [ ]:
DECISION = """
"""   # TODO
print(len(DECISION.split()), "palabras")

## 9. Para la práctica 12

El starter repite estas secciones con corridas repetidas, las tres políticas de contexto medidas y el reporte. Lleva contigo el paquete de la actividad 1 y el párrafo de la actividad 2.